### A. Install packages

In [ ]:
%pip install \
  --trusted-host nexus.tde.imperial.nhs.uk \
  --index-url http://nexus.tde.imperial.nhs.uk:8081/repository/pypi/simple/ \
  icare-risk duckdb snowflake-connector-python sqlalchemy

### B. Connect to snowflake
1. Open the URL shown below and wait a moment for the page to load.
2. An error "Hmmm.. cant reach this page" will appear. Copy the URL from the browser's address bar.
3. Paste it into the text field below, and click Submit.
4. If everything works correctly, you will see a "Login completed!" message.

In [ ]:
from icare_risk.clinphen.io.snowflake import get_snowflake_engine
from icare_risk.clinphen.io.snowflake import test_snowflake_connection

engine = get_snowflake_engine()
test_snowflake_connection(engine=engine)

### C. Download data
In this section, we download data from the Snowflake databases.

1. Executes an external SQL script (`cohort1.sql`) to retrieve the patient cohort metadata from the database.
2. Load saved .csv and extract the subject ids.
3. Pull all clinical records for those subject ids.

##### C.1 Fetch and Export Cohort Patient IDs
Run SQL extracting essential metadata (e.g. subject IDs, spell identifiers, encounter numbers, dates, ...) and saves the dataset locally as a CSV.

In [ ]:
# ----------------------------------------------------
# Download cohort (patient ids)
# ----------------------------------------------------
# .. note: Only run if you need to download the full data
#          for the ESBL patients. Otherwise, just continue
#          with the two sample pids provided below.

from sqlalchemy import text
from pathlib import Path

def fetch_dataframe_from_sql_query(engine, filepath):
    import pandas as pd
    with open(filepath, 'r') as f:
        sql_script = f.read()
    df = pd.read_sql(sql_script, engine)
    return df

# Define output directory
outpath = Path('data/cohorts')
outpath.mkdir(parents=True, exist_ok=True)

# Run SQL query
filepath = '../../data/scripts/cohort1.sql'
df = fetch_dataframe_from_sql_query(engine, filepath)

# Export to CSV
df.to_csv(outpath / 'esbl.csv')
display(df)

##### C.2 Load patient ids
Reads the saved cohort CSV file form disk (for inspection if needed).

In [ ]:
import pandas as pd

cohortpath = 'data/cohorts/esbl.csv'
cohort = pd.read_csv(cohortpath, usecols=['subject'])
target_pids = cohort.subject.unique().tolist()

print(f"Loaded {len(target_pids)} unique patient IDs.")

##### C.3 Download all clinical data for selected patients
Pull all corresponding clinical records for the filtered cohort from Snowflake

In [ ]:
# ----------------------------------------------------
# Download data for the cohort of patients selected
# ----------------------------------------------------
from icare_risk.clinphen.io.snowflake import download_and_save_tables
from icare_risk.clinphen.utils.io import check_lengths

# Load DataFrame
#df = pd.read_csv('../data/cohorts/ESBL_core_table.csv')

# Target
#target_pids = [
#    'ba094e447ce99b792aad017a66457d0beab29c6f7166dd2c54676f32fe0d1691',
#    'e31990ee52480feeea9ef380509f92909a19a67663aff2e0ef5b44a234e92b8f'
#]

# .. note: It takes a very long time.
# .. note: The vital signs file has 28GB!
# .. note: The pathology file has 12GB!
# .. note: This can be reduced by only downloading specific codes (e.g. rr, hr, crp, ...)
target_pids = pd.read_csv(cohortpath, usecols=['subject']) \
    .subject.unique().tolist()

# .. note: Either chunk false or very large chunk size.
tables_config = [
    {'table_name': 'icare_episodes_anon'},
    {'table_name': 'icare_problems_anon'},
    {'table_name': 'icare_vital_signs_anon'},
    {'table_name': 'icare_pathology_blood_anon'},
    {'table_name': 'icare_pharmacy_prescribing_anon'},
    {'table_name': 'icare_microbiology_anon'},
    {'table_name': 'icare_pharmacy_administration_anon'},
    {'table_name': 'icare_episodes_diagnosis_anon'},
    {'table_name': 'icare_episodes_procedures_anon'},
    {'table_name': 'icare_demographic_anon'},
]

FILE_FORMAT = 'parquet'

saved_dir = download_and_save_tables(
    tables=tables_config,
    output_dir='./data/raw/',
    engine=engine,
    id_list=target_pids,
    column_name='subject',
    file_format=FILE_FORMAT,
    verbose=False
)

check_lengths(data_dir=saved_dir, file_format=FILE_FORMAT)

### D. Download Lookup Tables

**Purpose:** Exports reference and mapping dictionaries from the database to decode clinical codes into human-readable descriptions.

**What it captures:** Extracts code definitions, names, units, value ranges, and multi-dimensional options.

**Output:** Saves the lookup tables as CSV files into a dedicated local directory.

In [ ]:
from icare_risk.clinphen.io.snowflake import export_lookup_counts
from icare_risk.clinphen.io.snowflake import export_lookup_counts_informative
from icare_risk.clinphen.io.snowflake import export_multidim_lookup

# Example A: Basic export lookup (just counts)
#export_lookup_counts(
#   engine=engine, table_name="ICARE_VITAL_SIGNS_ANON",
#   columns=["observation_code", "observation_name", "observation_unit"],
#   output_csv='./data/lookups/exmaple_a_vital_signs_lookup.csv'
#)

# Example B: Informative (with min, max and value_strings)
#export_lookup_counts_informative(
#   engine=engine, table_name="ICARE_VITAL_SIGNS_ANON",
#   code_col="observation_code", name_col="observation_name",
#   value_col="observation_result_clean", unit_col="observation_unit",
#   output_csv='./data/lookups/exmaple_b_vital_signs_lookup.csv'
#)

# Example C: Multidimensional (allows to keep 'pairs' of columns whose values go together)
# .. note: At the moment the multidimensional does not compute the low, high and value_strings,
#          hence, it is not useful for tables such as vital signs or pathology. This functionality
#          is pending to implement

# ----------------------------------------
# Download all lookups
# ----------------------------------------
DOWNLOAD_LOOKUPS = False

# Define path
path = Path('./data/lookups/')
# Ensure parent folder exists
path.mkdir(parents=True, exist_ok=True)

if DOWNLOAD_LOOKUPS:

    # Download
    export_lookup_counts_informative(
        engine=engine, table_name="ICARE_VITAL_SIGNS_ANON",
        code_col="observation_code", name_col="observation_name",
        value_col="observation_result_clean", unit_col="observation_unit",
        output_csv=path / "vital_signs_lookup.csv"
    )

    export_lookup_counts_informative(
        engine=engine, table_name="ICARE_PATHOLOGY_BLOOD_ANON",
        code_col="test_code", name_col="test_name",
        value_col="result_cleaned", unit_col="test_result_unit",
        output_csv=path / "pathology_blood_tests_lookup.csv"
    )

    export_lookup_counts_informative(
        engine=engine, table_name="ICARE_PROBLEMS_ANON",
        code_col="problem_code", name_col="problem_desc",
        value_col="problem_desc", unit_col=None,
        output_csv=path / "problems_lookup.csv"
    )

    export_lookup_counts_informative(
        engine=engine, table_name="ICARE_PHARMACY_PRESCRIBING_ANON",
        code_col="medication_name_short", name_col="medication_name_short",
        value_col="ordered_dose", unit_col="ordered_unit",
        output_csv=path / "pharmacy_prescribing_lookup.csv"
    )

    # Opt A: One row for each sensitivity
    export_multidim_lookup(
        engine=engine, table_name="ICARE_MICROBIOLOGY_ANON",
        group_cols=['ORDER_CODE', 'ORDER_NAME', 'SITE', 'ORGANISM_BUG', 'SENSITIVITY'],
        listagg_col=None, listagg_alias=None,
        output_csv=path / "microbiology_lookup.csv"
    )

    # Opt B: All options in string values
    export_multidim_lookup(
        engine=engine, table_name="ICARE_MICROBIOLOGY_ANON",
        group_cols=['ORDER_CODE', 'ORDER_NAME', 'SITE', 'ORGANISM_BUG'],
        listagg_col="SENSITIVITY", listagg_alias="sensitivity_options",
        output_csv=path / "microbiology_B_lookup.csv"
    )

    export_multidim_lookup(
        engine=engine, table_name="ICARE_EPISODES_DIAGNOSIS_ANON",
        group_cols=['DIAGNOSIS_CODE_ICD', 'DIAGNOSIS_CODE_SNOMED', 'DIAGNOSIS_DESC_ICD', 'DIAGNOSIS_DESC_SNOMED'],
        listagg_col=None, listagg_alias=None,
        output_csv=path / "episodes_diagnosis_B_lookup.csv"
    )

    export_multidim_lookup(
        engine=engine, table_name="ICARE_EPISODES_PROCEDURES_ANON",
        group_cols=['PROCEDURE_CODE_OPCS', 'PROCEDURE_CODE_SNOMED', 'PROCEDURE_NAME_OPCS', 'PROCEDURE_NAME_SNOMED'],
        listagg_col=None, listagg_alias=None,
        output_csv=path / "episodes_procedures_B_lookup.csv"
    )